# Migration Phase 3: Diffusion Tabular Derivatives (`TabularDerivativesLoader.load_diffusion`)

Replaces `DiffusionLoader` (QSIPrep/QSIRecon, on-the-fly parcellation). Diffusion
metrics are now pre-parcellated as TSVs under
`sub-<uid>/ses-<session_id>/dwi/atlas-<ATLAS_NAME>/`.

Each session has **one TSV per (software, model, param[, desc])** combination -
e.g. `software-DSIStudio_..._model-tensor_param-fa_diffmap.tsv`,
`software-AMICONODDI_..._model-noddi_param-icvf_desc-modulated_diffmap.tsv`.
`load_diffusion()` reads every `*_diffmap.tsv` for the requested sessions, parses
the BIDS entities from the filename via `parse_bids_entities()`, and concatenates
into one long-format table.

**Note**: diffusion derivatives only exist under `ses-<id>/` (plain), never
`ses-<id>.cross/` - `_dwi_atlas_dir()` checks both regardless of `SESSION_VARIANT`.

In [1]:
from pathlib import Path
from dotenv import load_dotenv
import os

load_dotenv(Path.cwd().parent / ".env")

from neuroalign.data.loaders import BehavioralLoader, TabularDerivativesLoader

beh = BehavioralLoader(os.environ["BRAINLINK_DB_PATH"])
loader = TabularDerivativesLoader(
    os.environ["TABULAR_DERIVATIVES_ROOT"],
    atlas_name=os.environ["ATLAS_NAME"],
    anat_atlases=tuple(os.environ["ANAT_ATLASES"].split(",")),
    session_variant=os.environ["SESSION_VARIANT"],
)

## 1. Find sessions with diffusion data

Each session has ~57 TSVs (one per software/model/param/desc combo) x 432
regions, so loading all ~4,900 sessions is expensive (Phase 5 pipeline run).
Here we use a 5-session sample, picking sessions that actually have a
`dwi/atlas-<ATLAS_NAME>` directory.

In [2]:
sessions = beh.get_sessions()
tabular_root = Path(os.environ["TABULAR_DERIVATIVES_ROOT"])
atlas = os.environ["ATLAS_NAME"]

found = []
for uid, sid in sessions[["uid", "session_id"]].drop_duplicates().itertuples(index=False):
    if (tabular_root / f"sub-{uid}" / f"ses-{sid}" / "dwi" / f"atlas-{atlas}").exists():
        found.append((uid, sid))
        if len(found) >= 5:
            break

import pandas as pd
sample = pd.DataFrame(found, columns=["uid", "session_id"])
sample

,uid,session_id
0,S629697,202601111959
1,S667793,202601181721
2,S307570,202601051400
3,S076379,202410131245
4,S105498,202601141203


## 2. Load diffusion data

In [3]:
dwi = loader.load_diffusion(sample)
print(f"{len(dwi)} rows")
print(dwi.columns.tolist())

123120 rows
['uid', 'session_id', 'index', 'label', 'hemisphere', 'volume_mm3', 'voxel_count', 'z_filtered_mean', 'z_filtered_std', 'iqr_filtered_mean', 'iqr_filtered_std', 'robust_mean', 'robust_std', 'mad_median', 'mean', 'std', 'median', 'sum', 'cv', 'robust_cv', 'skewness', 'excess_kurtosis', 'percentile_5', 'percentile_25', 'percentile_75', 'percentile_95', 'coverage', 'scalar', 'atlas', 'software', 'model', 'param', 'desc']


## 3. software / model / param / desc combinations

432 regions x 5 sessions = 2160 rows per (software, model, param, desc) combo.

In [4]:
combos = dwi.groupby(["software", "model", "param", "desc"], dropna=False).size()
n_sessions = sample.shape[0]
print(f"{len(combos)} combos x 432 regions x {n_sessions} sessions = {combos.sum()} (= {dwi.shape[0]})")
print((combos == 432 * n_sessions).all(), "-> every combo has exactly 432 * n_sessions region rows")
combos.head(10)

57 combos x 432 regions x 5 sessions = 123120 (= 123120)
True -> every combo has exactly 432 * n_sessions region rows


software    model  param      desc     
AMICONODDI  noddi  direction  NaN          2160
                   icvf       modulated    2160
                              NaN          2160
                   isovf      NaN          2160
                   nrmse      NaN          2160
                   od         modulated    2160
                              NaN          2160
                   rmse       NaN          2160
                   tf         NaN          2160
DIPYDKI     dki    ad         NaN          2160
dtype: int64

## 4. Region labels

`LH_*`/`RH_*` (Schaefer cortex, no `7Networks_` prefix - see Phase 2 note) +
`*-lh`/`*-rh` (Tian2020S2 subcortex, matches anat exactly).

In [5]:
one_combo = dwi[
    (dwi["uid"] == sample["uid"].iloc[0])
    & (dwi["session_id"] == sample["session_id"].iloc[0])
    & (dwi["software"] == "DSIStudio")
    & (dwi["model"] == "tensor")
    & (dwi["param"] == "fa")
]
print(f"{len(one_combo)} regions")
print("cortex sample:", one_combo[one_combo["label"].str.contains("_")]["label"].head(3).tolist())
print("subcortex sample:", one_combo[one_combo["label"].str.contains("-")]["label"].head(3).tolist())
one_combo[["label", "hemisphere", "mean", "std", "median", "scalar"]].head()

432 regions
cortex sample: ['LH_Vis_1', 'LH_Vis_2', 'LH_Vis_3']
subcortex sample: ['aHIP-rh', 'pHIP-rh', 'lAMY-rh']


,label,hemisphere,mean,std,median,scalar
18576,LH_Vis_1,L,0.192400,0.099598,0.167911,fa
18577,LH_Vis_2,L,0.241685,0.131431,0.209075,fa
18578,LH_Vis_3,L,0.220690,0.127093,0.191007,fa
18579,LH_Vis_4,L,0.222666,0.123165,0.193205,fa
18580,LH_Vis_5,L,0.258495,0.145190,0.229208,fa
